# Import Required Libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


import joblib

import warnings
warnings.filterwarnings("ignore")

In [11]:
column_names = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income"
]

# Load Dataset

In [12]:
# ==========================================================
# Load Training Dataset
# ==========================================================

train_df = pd.read_csv(
    "data/adult_train.csv",
    header=None,
    names=column_names,
    na_values="?",
    skipinitialspace=True
)

print("Training Dataset Loaded Successfully!")

Training Dataset Loaded Successfully!


In [13]:
# ==========================================================
# Load Test Dataset
# ==========================================================

test_df = pd.read_csv(
    "data/adult_test.csv",
    header=None,
    names=column_names,
    na_values="?",
    skiprows=1,
    skipinitialspace=True
)

print("Testing Dataset Loaded Successfully!")

Testing Dataset Loaded Successfully!


In [14]:
# -----------------------------
# 1. Remove Duplicate Records
# -----------------------------

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()


# -----------------------------
# 2. Remove Missing Values
# -----------------------------

train_df = train_df.dropna()
test_df = test_df.dropna()

In [15]:
# 3. Clean Target Variable

train_df["income"] = train_df["income"].str.replace(".", "", regex=False)
test_df["income"] = test_df["income"].str.replace(".", "", regex=False)


# 4. Separate Features & Target

X_train = train_df.drop("income", axis=1)
y_train = train_df["income"]

X_test = test_df.drop("income", axis=1)
y_test = test_df["income"]


# One-Hot Encode Features

In [16]:
X_train = pd.get_dummies(
    X_train,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    drop_first=True
)

In [17]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)


In [18]:
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)


print("Preprocessing Completed Successfully!")

Preprocessing Completed Successfully!


# Feature Scaling

In [19]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Feature Scaling Completed Successfully!")

Feature Scaling Completed Successfully!


# Split Training Data into Training and Validation Sets

In [20]:
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_scaled,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

In [21]:
print("Training Set :", X_train_final.shape)

print("Validation Set :", X_val.shape)

print("Testing Set :", X_test_scaled.shape)

Training Set : (24111, 96)
Validation Set : (6028, 96)
Testing Set : (15055, 96)


# Baseline Model Training

In [22]:
import time

In [23]:
baseline_results = {
    "Model": [],
    "Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1 Score": [],
    "Training Time (s)": []
}

# Train the Random Forest Baseline

In [24]:
rf = RandomForestClassifier(random_state=42)

start_time = time.time()

rf.fit(X_train_final, y_train_final)

training_time = time.time() - start_time

rf_predictions = rf.predict(X_val)

baseline_results["Model"].append("Random Forest")

baseline_results["Accuracy"].append(
    accuracy_score(y_val, rf_predictions)
)

baseline_results["Precision"].append(
    precision_score(y_val, rf_predictions)
)

baseline_results["Recall"].append(
    recall_score(y_val, rf_predictions)
)

baseline_results["F1 Score"].append(
    f1_score(y_val, rf_predictions)
)

baseline_results["Training Time (s)"].append(training_time)

print("Random Forest Completed!")

Random Forest Completed!


# Train the Gradient Boosting Baseline

In [25]:
gb = GradientBoostingClassifier(random_state=42)

start_time = time.time()

gb.fit(X_train_final, y_train_final)

training_time = time.time() - start_time

gb_predictions = gb.predict(X_val)

baseline_results["Model"].append("Gradient Boosting")

baseline_results["Accuracy"].append(
    accuracy_score(y_val, gb_predictions)
)

baseline_results["Precision"].append(
    precision_score(y_val, gb_predictions)
)

baseline_results["Recall"].append(
    recall_score(y_val, gb_predictions)
)

baseline_results["F1 Score"].append(
    f1_score(y_val, gb_predictions)
)

baseline_results["Training Time (s)"].append(training_time)

print("Gradient Boosting Completed!")

Gradient Boosting Completed!


# Train the SVM Baseline

In [26]:
svm = SVC(random_state=42)

start_time = time.time()

svm.fit(X_train_final, y_train_final)

training_time = time.time() - start_time

svm_predictions = svm.predict(X_val)

baseline_results["Model"].append("Support Vector Machine")

baseline_results["Accuracy"].append(
    accuracy_score(y_val, svm_predictions)
)

baseline_results["Precision"].append(
    precision_score(y_val, svm_predictions)
)

baseline_results["Recall"].append(
    recall_score(y_val, svm_predictions)
)

baseline_results["F1 Score"].append(
    f1_score(y_val, svm_predictions)
)

baseline_results["Training Time (s)"].append(training_time)

print("Support Vector Machine Completed!")

Support Vector Machine Completed!


In [27]:
# ==========================================================
# Baseline Model Performance
# ==========================================================

baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,Random Forest,0.851194,0.738924,0.622252,0.675588,2.903257
1,Gradient Boosting,0.859821,0.787215,0.598934,0.680288,6.481229
2,Support Vector Machine,0.844393,0.747147,0.566955,0.644697,29.898150


# Define Hyperparameter Distribution

In [28]:
gb_param_dist = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.8, 1.0]
}

In [ ]:
# Randomized Search - Gradient Boosting

gb_random = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=10,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_random.fit(X_train_final, y_train_final)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [32]:
print(gb_random.best_params_)
print(gb_random.best_score_)

{'subsample': 1.0, 'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 5, 'learning_rate': 0.1}
0.8691053247637457


# Random Forest Hyperparameter Tuning

In [33]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

In [34]:
# Grid Search - Random Forest

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=2
)

rf_grid.fit(X_train_final, y_train_final)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'max_depth': [10, 20, ...], 'min_samples_leaf': [1, 2], 'min_samples_split': [2, 5], 'n_estimators': [100, 200]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [35]:
print("Best Parameters:")
print(rf_grid.best_params_)

print()

print("Best Cross Validation Accuracy:")
print(rf_grid.best_score_)

Best Parameters:
{'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}

Best Cross Validation Accuracy:
0.8599811080821856


# Evaluate on Validation Set

In [40]:
rf_best = rf_grid.best_estimator_

rf_val_predictions = rf_best.predict(X_val)

print(classification_report(y_val, rf_val_predictions))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91      4527
           1       0.79      0.60      0.68      1501

    accuracy                           0.86      6028
   macro avg       0.83      0.77      0.80      6028
weighted avg       0.85      0.86      0.85      6028



# Gradient Boosting Hyperparameter Tuning

In [36]:
gb_param_dist = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.8, 1.0]
}

In [ ]:
# Randomized Search - Gradient Boosting

gb_random = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_distributions=gb_param_dist,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

gb_random.fit(X_train_final, y_train_final)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [41]:
print("Best Parameters:")
print(gb_random.best_params_)

print()

print("Best Cross Validation Accuracy:")
print(gb_random.best_score_)

Best Parameters:
{'subsample': 1.0, 'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_depth': 7, 'learning_rate': 0.05}

Best Cross Validation Accuracy:
0.8691054279606748


# Evaluate on Validation Set

In [42]:
gb_best = gb_random.best_estimator_

gb_predictions = gb_best.predict(X_val)

print(classification_report(y_val, gb_predictions))

              precision    recall  f1-score   support

           0       0.89      0.94      0.91      4527
           1       0.77      0.65      0.70      1501

    accuracy                           0.86      6028
   macro avg       0.83      0.79      0.81      6028
weighted avg       0.86      0.86      0.86      6028



# SVM Hyperparameter Tuning

In [44]:
svm_param_dist = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}

In [ ]:
# Randomized Search - Support Vector Machine

svm_random = RandomizedSearchCV(
    estimator=SVC(random_state=42),
    param_distributions=svm_param_dist,
    n_iter=8,
    scoring="accuracy",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

svm_random.fit(X_train_final, y_train_final)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


In [47]:
print("Best Parameters:")
print(svm_random.best_params_)

print()

print("Best Cross Validation Accuracy:")
print(svm_random.best_score_)

Best Parameters:
{'kernel': 'linear', 'gamma': 'scale', 'C': 0.1}

Best Cross Validation Accuracy:
0.8457965244079467


# Evaluate on Validation Set

In [48]:
svm_best = svm_random.best_estimator_

svm_predictions = svm_best.predict(X_val)

print(classification_report(y_val, svm_predictions))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90      4527
           1       0.74      0.58      0.65      1501

    accuracy                           0.85      6028
   macro avg       0.81      0.76      0.78      6028
weighted avg       0.84      0.85      0.84      6028



# Evaluate Tuned Models on Test Set

# Random Forest

In [49]:
rf_test_predictions = rf_grid.best_estimator_.predict(X_test_scaled)

rf_accuracy = accuracy_score(y_test, rf_test_predictions)
rf_precision = precision_score(y_test, rf_test_predictions)
rf_recall = recall_score(y_test, rf_test_predictions)
rf_f1 = f1_score(y_test, rf_test_predictions)

print(classification_report(y_test, rf_test_predictions))

              precision    recall  f1-score   support

           0       0.88      0.95      0.91     11355
           1       0.78      0.60      0.68      3700

    accuracy                           0.86     15055
   macro avg       0.83      0.77      0.79     15055
weighted avg       0.85      0.86      0.85     15055



# Gradient Boosting

In [50]:
gb_test_predictions = gb_random.best_estimator_.predict(X_test_scaled)

gb_accuracy = accuracy_score(y_test, gb_test_predictions)
gb_precision = precision_score(y_test, gb_test_predictions)
gb_recall = recall_score(y_test, gb_test_predictions)
gb_f1 = f1_score(y_test, gb_test_predictions)

print(classification_report(y_test, gb_test_predictions))

              precision    recall  f1-score   support

           0       0.89      0.94      0.91     11355
           1       0.77      0.66      0.71      3700

    accuracy                           0.87     15055
   macro avg       0.83      0.80      0.81     15055
weighted avg       0.86      0.87      0.86     15055



# SVM

In [51]:
svm_test_predictions = svm_random.best_estimator_.predict(X_test_scaled)

svm_accuracy = accuracy_score(y_test, svm_test_predictions)
svm_precision = precision_score(y_test, svm_test_predictions)
svm_recall = recall_score(y_test, svm_test_predictions)
svm_f1 = f1_score(y_test, svm_test_predictions)

print(classification_report(y_test, svm_test_predictions))

              precision    recall  f1-score   support

           0       0.87      0.93      0.90     11355
           1       0.74      0.58      0.65      3700

    accuracy                           0.85     15055
   macro avg       0.80      0.76      0.78     15055
weighted avg       0.84      0.85      0.84     15055



# Comparative Performance Table

In [52]:
comparison_df = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Gradient Boosting",
        "Support Vector Machine"
    ],
    "Accuracy": [
        rf_accuracy,
        gb_accuracy,
        svm_accuracy
    ],
    "Precision": [
        rf_precision,
        gb_precision,
        svm_precision
    ],
    "Recall": [
        rf_recall,
        gb_recall,
        svm_recall
    ],
    "F1 Score": [
        rf_f1,
        gb_f1,
        svm_f1
    ]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.859249,0.780021,0.595135,0.675149
1,Gradient Boosting,0.867552,0.769937,0.657568,0.709329
2,Support Vector Machine,0.845699,0.735063,0.581892,0.649570


# Final Model Selection

In [55]:
if best_model_name == "Random Forest":
    best_model = rf_grid.best_estimator_

elif best_model_name == "Gradient Boosting":
    best_model = gb_random.best_estimator_

else:
    best_model = svm_random.best_estimator_

In [56]:
joblib.dump(best_model, "best_model.pkl")

print("Best model saved successfully!")

Best model saved successfully!
